# 🎭 Convolutional VAE — CelebA Face Generation

**Dataset:** CelebA — 202,599 RGB celebrity face images, resized to 128×128  
**Task:** Learn a structured latent space of faces — reconstruct, generate, and interpolate

## From Autoencoder → Variational Autoencoder

| Property | Autoencoder | VAE |
|---|---|---|
| Latent representation | Single point z | Distribution N(μ, σ²) |
| Sampling | Deterministic | Stochastic via reparameterization |
| Loss | Reconstruction only | Reconstruction + KL divergence |
| Latent space | Unstructured | Smooth and continuous |
| Generation quality | Poor (gaps in space) | Good (interpolation works) |

## The ELBO Objective

VAE maximizes the Evidence Lower Bound (ELBO):

$$\mathcal{L} = \underbrace{\mathbb{E}_{q(z|x)}[\log p(x|z)]}_{\text{Reconstruction}} - \underbrace{\beta \cdot D_{KL}(q(z|x) \| p(z))}_{\text{KL Divergence}}$$

**Reconstruction term** — how well the decoder rebuilds the input:
$$\mathcal{L}_{recon} = -\sum_{i} [x_i \log \hat{x}_i + (1-x_i)\log(1-\hat{x}_i)]$$

**KL Divergence** — how close the learned distribution is to N(0,1):
$$\mathcal{L}_{KL} = -\frac{1}{2} \sum_{j} \left(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

## The Reparameterization Trick

Sampling $z \sim \mathcal{N}(\mu, \sigma^2)$ is not differentiable — gradients cannot flow through a sampling node.

**Fix:** Rewrite as a deterministic function:
$$z = \mu + \sigma \cdot \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, 1)$$

$\varepsilon$ is sampled independently — gradients flow through $\mu$ and $\sigma$, not through the sampling step.

## Architecture

```
Encoder: (B,3,128,128) → Conv×4 (stride=2) → Flatten → fc_mu + fc_log_var → (B,128)
Decoder: (B,128) → Linear → Reshape → ConvTranspose×4 (stride=2) → Sigmoid → (B,3,128,128)
```

## Beta-VAE

Setting $\beta > 1$ encourages more disentangled latent dimensions at the cost of reconstruction quality:
- $\beta = 1$ → Standard VAE — best reconstruction  
- $\beta = 4$ → More disentangled — each latent dim controls one factor (pose, lighting, smile)  
- $\beta = 8$ → Highly disentangled — blurry reconstructions

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import CelebA
from torch.utils.data import DataLoader
from PIL import Image

from src.encoder import Encoder
from src.decoder import Decoder
from src.vae     import ConvVAE
from src.trainer import VAETrainer

with open('../configs/vae_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Config loaded:')
for section, values in config.items():
    print(f'  {section}: {values}')
print(f'\nDevice: {device}')

In [ ]:
# Inspect model architecture and verify forward pass shapes
model = ConvVAE(
    latent_dim     = config['model']['latent_dim'],
    base_channels  = config['model']['base_channels'],
    image_channels = config['model']['image_channels'],
    image_size     = config['model']['image_size'],
    beta           = config['model']['beta'],
).to(device)

print('Encoder:')
print(model.encoder)
print(f'\nDecoder:')
print(model.decoder)
print(f'\nTotal parameters: {model.count_parameters():,}')

# verify forward pass shapes
x_test               = torch.rand(4, 3, 128, 128).to(device)
x_recon, mu, log_var = model(x_test)
z_test               = model.reparameterize(mu, log_var)

print(f'\nShape check:')
print(f'  Input        : {x_test.shape}')
print(f'  mu           : {mu.shape}')
print(f'  log_var      : {log_var.shape}')
print(f'  z            : {z_test.shape}')
print(f'  Reconstruction: {x_recon.shape}')
print(f'  Output range : [{x_recon.min():.3f}, {x_recon.max():.3f}]  ← should be [0,1]')

In [ ]:
# Train the model
# CPU  : ~30-40 mins per epoch at batch_size=32
# GPU  : ~3-5 mins per epoch
# Increase epochs to 30+ on GPU for quality face generation

trainer = VAETrainer(config)
trainer.train()

history = trainer.get_history()
print(f'\nTraining complete.')
print(f'Final Total Loss : {history["total_losses"][-1]:.2f}')
print(f'Final Recon Loss : {history["recon_losses"][-1]:.2f}')
print(f'Final KL Loss    : {history["kl_losses"][-1]:.2f}')

In [ ]:
# Plot all three loss curves separately
history = trainer.get_history()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, data, color, title, note in zip(
    axes,
    [history['total_losses'], history['recon_losses'], history['kl_losses']],
    ['#9B59B6',               '#E74C3C',               '#1A6BC4'],
    ['Total Loss (ELBO)',      'Reconstruction Loss',   'KL Divergence'],
    ['Should decrease',        'Should decrease',       'Should stabilize > 0'],
):
    ax.plot(data, color=color, linewidth=1.8)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_title(title,    fontsize=12)
    ax.annotate(note, xy=(0.5, 0.04), xycoords='axes fraction',
                ha='center', fontsize=9, color='gray')
    ax.grid(alpha=0.3)

plt.suptitle('Convolutional VAE — Training Curves (CelebA)', fontsize=14)
plt.tight_layout()
plt.show()

print('\nHealthy training signs:')
print('  Total loss   : steadily decreasing')
print('  Recon loss   : decreasing — model learning to reconstruct')
print('  KL divergence: stabilizes at a positive value (not collapsing to 0)')
print('  KL collapse  : if KL → 0, decoder ignores latent space (posterior collapse)')

In [ ]:
# Compare real vs reconstructed faces
# Top row: original faces
# Bottom row: VAE reconstructions

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])
dataset = CelebA(root='../data', split='valid', transform=transform, download=True)
loader  = DataLoader(dataset, batch_size=8, shuffle=True)
real, _ = next(iter(loader))

trainer.model.eval()
with torch.no_grad():
    recon, _, _ = trainer.model(real.to(trainer.device))

# interleave: real1, recon1, real2, recon2...
comparison = torch.cat([real.to(trainer.device), recon], dim=0)
grid = torchvision.utils.make_grid(comparison, nrow=8, normalize=False)
grid_np = grid.permute(1, 2, 0).cpu().numpy()

fig, ax = plt.subplots(figsize=(16, 5))
ax.imshow(grid_np)
ax.axis('off')
ax.set_title('Top: Original CelebA faces   |   Bottom: VAE Reconstructions', fontsize=13)
plt.tight_layout()
os.makedirs('../assets', exist_ok=True)
plt.savefig('../assets/reconstruction_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → assets/reconstruction_comparison.png')

In [ ]:
# Generate new faces by sampling z ~ N(0,1)
# These are completely new faces — not in the training set

trainer.model.eval()
with torch.no_grad():
    samples = trainer.model.sample(16, trainer.device)   # (16, 3, 128, 128)

grid    = torchvision.utils.make_grid(samples, nrow=4, normalize=False)
grid_np = grid.permute(1, 2, 0).cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(grid_np)
ax.axis('off')
ax.set_title('Generated Faces — sampled from z ~ N(0,1)', fontsize=13)
plt.tight_layout()
plt.savefig('../assets/generated_faces.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → assets/generated_faces.png')

In [ ]:
# Interpolate between two faces in latent space
# This only works because VAE latent space is continuous and structured
# A regular autoencoder would produce broken images between two points

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])
dataset    = CelebA(root='../data', split='valid', transform=transform, download=False)
loader     = DataLoader(dataset, batch_size=2, shuffle=True)
two_faces, _ = next(iter(loader))

trainer.model.eval()
with torch.no_grad():
    mu, log_var = trainer.model.encoder(two_faces.to(trainer.device))
    z1, z2      = mu[0], mu[1]     # use mu for deterministic interpolation

    # linearly interpolate between z1 and z2
    steps  = 8
    alphas = torch.linspace(0, 1, steps)
    interp_imgs = []

    for alpha in alphas:
        z_interp = (1 - alpha) * z1 + alpha * z2
        img      = trainer.model.decoder(z_interp.unsqueeze(0))
        interp_imgs.append(img)

    interp_imgs = torch.cat(interp_imgs, dim=0)  # (8, 3, 128, 128)

grid    = torchvision.utils.make_grid(interp_imgs, nrow=8, normalize=False)
grid_np = grid.permute(1, 2, 0).cpu().numpy()

fig, ax = plt.subplots(figsize=(20, 4))
ax.imshow(grid_np)
ax.axis('off')
ax.set_title('Latent Space Interpolation — Face A → Face B', fontsize=13)
plt.tight_layout()
plt.savefig('../assets/latent_interpolation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → assets/latent_interpolation.png')
print('\nNote: smooth transitions = structured latent space')
print('      broken images = posterior collapse or undertrained model')

In [ ]:
# Visualize latent space structure using t-SNE
# CelebA has 40 binary attributes — color by 'Smiling' attribute (index 31)
# If VAE learned meaningful features, smiling/non-smiling faces should cluster separately

from sklearn.manifold import TSNE

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])
dataset = CelebA(root='../data', split='valid', transform=transform, download=False)
loader  = DataLoader(dataset, batch_size=64, shuffle=True)

all_mu, all_labels = [], []
trainer.model.eval()

# collect 500 samples for t-SNE
with torch.no_grad():
    for imgs, attrs in loader:
        mu, _ = trainer.model.encoder(imgs.to(trainer.device))
        all_mu.append(mu.cpu())
        all_labels.append(attrs[:, 31])  # 31 = Smiling attribute
        if len(all_mu) * 64 >= 500:
            break

all_mu     = torch.cat(all_mu)[:500].numpy()
all_labels = torch.cat(all_labels)[:500].numpy()

# t-SNE projection: 128-dim → 2-dim
print('Running t-SNE (this takes ~1 minute)...')
tsne      = TSNE(n_components=2, random_state=42, perplexity=30)
mu_2d     = tsne.fit_transform(all_mu)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    mu_2d[:, 0], mu_2d[:, 1],
    c=all_labels, cmap='coolwarm',
    alpha=0.6, s=15,
)
plt.colorbar(scatter, ax=ax, label='Smiling (0=No, 1=Yes)')
ax.set_title('t-SNE of VAE Latent Space — colored by Smiling attribute', fontsize=13)
ax.set_xlabel('t-SNE dim 1')
ax.set_ylabel('t-SNE dim 2')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../assets/tsne_latent_space.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → assets/tsne_latent_space.png')
print('\nGood sign: smiling (red) and non-smiling (blue) clusters are separated')

In [ ]:
# Beta-VAE experiment — compare reconstruction quality at different beta values
# Uses the SAME trained checkpoint, just changes the beta weight at inference
# To see the full effect: retrain with beta=4 in vae_config.yaml

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])
dataset    = CelebA(root='../data', split='valid', transform=transform, download=False)
loader     = DataLoader(dataset, batch_size=4, shuffle=True)
real, _    = next(iter(loader))

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

betas = [1.0, 4.0, 8.0]
labels = ['β=1 (Standard VAE)', 'β=4 (Beta-VAE)', 'β=8 (High disentanglement)']

trainer.model.eval()
for row, (beta, label) in enumerate(zip(betas, labels)):
    trainer.model.beta = beta
    with torch.no_grad():
        recon, mu, log_var = trainer.model(real.to(trainer.device))
        _, recon_loss, kl_loss = trainer.model.loss(real.to(trainer.device), recon, mu, log_var)

    for col in range(4):
        img = recon[col].cpu().permute(1, 2, 0).numpy()
        axes[row, col].imshow(img)
        axes[row, col].axis('off')

    axes[row, 0].set_ylabel(
        f'{label}\nRecon={recon_loss.item():.1f} KL={kl_loss.item():.1f}',
        fontsize=9, rotation=0, labelpad=120, va='center'
    )

# reset beta to config value
trainer.model.beta = config['model']['beta']

plt.suptitle('Beta-VAE Experiment — Reconstruction at Different β Values', fontsize=13)
plt.tight_layout()
plt.savefig('../assets/beta_vae_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → assets/beta_vae_comparison.png')
print('\nExpected: higher beta = blurrier reconstructions but more disentangled latent space')

## Results Summary

| Metric | Value |
|--------|-------|
| Final Total Loss | *see training output* |
| Final Recon Loss | *see training output* |
| Final KL Loss | *see training output* |
| Training Epochs | see config |
| Latent Dim | 128 |
| Beta | 1.0 |

## Key Takeaways

**Reparameterization trick** is what makes VAE trainable end-to-end.  
Without it, gradients cannot flow through the sampling step.

**KL collapse (posterior collapse)** is the main failure mode to watch for.  
If KL → 0, the decoder learned to ignore z entirely — reconstructions look average/blurry.

**Beta tradeoff**: higher beta forces more disentangled latent dimensions  
at the cost of reconstruction sharpness. β=1 maximizes reconstruction quality.

**Latent interpolation** works smoothly because the KL term enforces  
a continuous, gap-free latent space — unlike a regular autoencoder.

**t-SNE clustering** shows whether the model learned semantically  
meaningful structure — smiling vs non-smiling faces should separate  
naturally without any supervision on that attribute.